# Cinema Backend — Kaggle CPU baseline

Commit: 36c8922. Run Movie Read and Booking at 10/25/50 VUs, three repetitions, 30s warm-up and 120s measurement. API, PostgreSQL and k6 share this CPU session. Contention is excluded pending the documented contract fixes.

In [ ]:
import os, pathlib, subprocess, json, time, urllib.request

repo = pathlib.Path('/kaggle/working/BTL-KTPM')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'dev',
                    'https://github.com/Huy-hoc-it/BTL-KTPM.git', str(repo)], check=True)
os.chdir(repo)
# Để tái lập đúng bản đã đối chiếu; đổi SHA nếu nhóm đã sửa và push phiên bản mới.
subprocess.run(['git', 'checkout', '36c8922'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y openjdk-21-jdk-headless postgresql postgresql-contrib nodejs curl ca-certificates gnupg
curl -fsSL https://dl.k6.io/key.gpg | sudo gpg --batch --yes --dearmor -o /usr/share/keyrings/k6-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/k6-archive-keyring.gpg] https://dl.k6.io/deb stable main' | sudo tee /etc/apt/sources.list.d/k6.list
sudo apt-get update -qq
sudo apt-get install -y k6


In [ ]:
import glob
jdk = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk-*'))
assert jdk, 'Chưa cài được JDK 21'
os.environ['JAVA_HOME'] = jdk[0]
os.environ['PATH'] = jdk[0] + '/bin:' + os.environ['PATH']
subprocess.run(['java', '-version'], check=True)
subprocess.run(['k6', 'version'], check=True)
node_major = int(subprocess.check_output(['node', '-p', 'process.versions.node.split(".")[0]'], text=True))
assert node_major >= 18, 'Seed script cần Node.js >= 18 (fetch API)'


In [ ]:
clusters = subprocess.check_output(['pg_lsclusters', '--no-header'], text=True).splitlines()
assert clusters, 'Gói PostgreSQL chưa tạo cluster; kiểm tra quá trình cài đặt'
cluster = clusters[-1].split()
pg_version, pg_name, pg_port = cluster[:3]
subprocess.run(['sudo', 'pg_ctlcluster', '--skip-systemctl-redirect',
                pg_version, pg_name, 'start'], check=False)

def pg(sql):
    return subprocess.check_output(['sudo', '-u', 'postgres', 'psql', '-p', pg_port,
                                    '-v', 'ON_ERROR_STOP=1', '-tAc', sql], text=True).strip()

pg("""DO $$ BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname='cinema_load') THEN
        CREATE ROLE cinema_load LOGIN PASSWORD 'cinema_load_pw';
    END IF;
END $$;""")
if pg("SELECT 1 FROM pg_database WHERE datname='cinema_load'") != '1':
    subprocess.run(['sudo', '-u', 'postgres', 'createdb', '-p', pg_port,
                    '-O', 'cinema_load', 'cinema_load'], check=True)
data_dir = pg('SHOW data_directory')
database_pid = int(subprocess.check_output(['sudo', 'head', '-n', '1', data_dir + '/postmaster.pid'], text=True))
print('PostgreSQL:', pg('SELECT version()'))
print('Database PID:', database_pid, 'port:', pg_port)


In [ ]:
os.environ.update({
    'SPRING_PROFILES_ACTIVE': 'local',
    'DB_URL': f'jdbc:postgresql://127.0.0.1:{pg_port}/cinema_load',
    'DB_USERNAME': 'cinema_load', 'DB_PASSWORD': 'cinema_load_pw',
    'JWT_SECRET': 'kaggle-local-only-cinema-benchmark-secret-2026-please-change',
    'DEMO_ADMIN_USERNAME': 'admin', 'DEMO_ADMIN_PASSWORD': 'demo-pass-1234',
    'JWT_EXPIRATION_SECONDS': '3600',
    'SERVER_ADDRESS': '127.0.0.1', 'SERVER_PORT': '8080',
    'BASE_URL': 'http://127.0.0.1:8080',
    'ADMIN_USERNAME': 'admin', 'ADMIN_PASSWORD': 'demo-pass-1234',
})
subprocess.run(['bash', './mvnw', '-B', '-DskipTests', 'package'], check=True)
jars = list((repo / 'target').glob('*.jar'))
assert len(jars) == 1, jars
api_log = open('/kaggle/working/cinema-api.log', 'w')
api = subprocess.Popen(['java', '-Xms256m', '-Xmx768m', '-jar', str(jars[0])],
                       stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy())
os.environ['API_PID'] = str(api.pid)
os.environ['DATABASE_PID'] = str(database_pid)
ready = False
for _ in range(120):
    if api.poll() is not None:
        break
    try:
        with urllib.request.urlopen(os.environ['BASE_URL'] + '/actuator/health/readiness', timeout=2) as r:
            ready = r.status == 200
        if ready:
            break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    print(pathlib.Path('/kaggle/working/cinema-api.log').read_text()[-12000:])
    raise RuntimeError('API chưa sẵn sàng; xử lý lỗi trước khi đo')
print('API ready, PID:', api.pid)


In [ ]:
import urllib.error
for method, route in [('GET', '/api/v1/users/me'), ('POST', '/api/v1/bookings')]:
    request = urllib.request.Request(os.environ['BASE_URL'] + route,
        data=b'{}' if method == 'POST' else None, method=method,
        headers={'Content-Type': 'application/json'})
    try:
        urllib.request.urlopen(request)
        raise AssertionError(f'{method} {route} cho truy cập khi thiếu token')
    except urllib.error.HTTPError as e:
        assert e.code == 401, (method, route, e.code)
        print(method, route, '→ 401 OK')

os.environ.update({'MOVIE_COUNT': '1000', 'DELETED_COUNT': '50',
                   'SEED_SCOPE': 'all', 'VUS': '50',
                   'FIXTURES': str(repo / 'load-test/fixtures.local.json')})
subprocess.run(['node', 'load-test/seed.mjs'], check=True)
subprocess.run(['k6', 'run', '--include-system-env-vars', 'load-test/smoke.js'], check=True)


In [ ]:
for scenario in ['movie-read', 'booking']:
    subprocess.run(['python', 'load-test/run-baseline.py', '--scenario', scenario,
                    '--api-pid', str(api.pid), '--database-pid', str(database_pid)], check=True)


In [ ]:
# Chưa chạy với commit 36c8922 vì các lỗi contract đã nêu.
# subprocess.run(['python', 'load-test/run-baseline.py', '--scenario', 'contention',
#                 '--api-pid', str(api.pid), '--database-pid', str(database_pid)], check=True)


In [ ]:
import csv, shutil
rows = []
for folder in sorted((repo / 'load-test/results').iterdir()):
    if not folder.is_dir():
        continue
    for f in sorted(folder.glob('*.json')):
        if f.name.endswith('-raw.json') or f.name == 'environment.json':
            continue
        result = json.loads(f.read_text())
        if 'metadata' not in result or 'summary' not in result:
            continue
        meta = result['metadata']
        metrics = result['summary']['metrics']
        values = lambda key: metrics.get(key, {}).get('values', {})
        rows.append({
            'run': folder.name + '/' + f.stem,
            'scenario': meta['script'], 'vus': meta['vus'],
            'req_per_s': meta.get('measuredThroughput'),
            'read_p50_ms': values('measured_read_latency').get('med'),
            'read_p95_ms': values('measured_read_latency').get('p(95)'),
            'read_p99_ms': values('measured_read_latency').get('p(99)'),
            'booking_p95_ms': values('measured_booking_latency').get('p(95)'),
            'technical_error_rate': values('measured_technical_error').get('rate'),
            'business_error_rate': values('measured_business_error').get('rate'),
        })
if rows:
    with open('/kaggle/working/kaggle-summary.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)

# Chỉ đóng gói kết quả, không đóng gói fixtures.local.json hay cấu hình mật khẩu.
archive = shutil.make_archive('/kaggle/working/cinema-kaggle-results', 'zip',
                              root_dir=str(repo / 'load-test'), base_dir='results')
print('Tải tại Output:', archive)
